# PairwiseGP

This notebook demonstrates `robotorchan.models.PairwiseGP` for preference learning from pairwise comparisons.

Unlike ordinary regression, the observations are comparisons such as *item i is preferred to item j* rather than scalar target values.

## 1. When to use this model

Use `PairwiseGP` when users, experts, or experiments provide relative preference information more naturally or reliably than absolute scores. Examples include human preference optimization, sensory evaluation, ranking, and qualitative materials screening.

In [ ]:
import matplotlib.pyplot as plt
import torch
from botorch.fit import fit_gpytorch_mll

from robotorchan.models import PairwiseGP

torch.manual_seed(0)
dtype = torch.double

## 2. Synthetic latent utility and pairwise comparisons

In [ ]:
def latent_utility(x: torch.Tensor) -> torch.Tensor:
    return torch.sin(2.0 * torch.pi * x).squeeze(-1) + 0.35 * x.squeeze(-1)

n_items = 18
datapoints = torch.linspace(0.0, 1.0, n_items, dtype=dtype).unsqueeze(-1)
utility = latent_utility(datapoints)

n_comparisons = 50
pairs = torch.randint(0, n_items, (n_comparisons, 2))
pairs = pairs[pairs[:, 0] != pairs[:, 1]]

comparisons = []
for i, j in pairs.tolist():
    # PairwiseGP expects each row as [winner, loser].
    if utility[i] >= utility[j]:
        comparisons.append([i, j])
    else:
        comparisons.append([j, i])

comparisons = torch.tensor(comparisons, dtype=torch.long)
datapoints.shape, comparisons.shape

## 3. Model construction and robotorchan API

In [ ]:
model = PairwiseGP(
    datapoints=datapoints,
    comparisons=comparisons,
)

print("raw_datapoints:", model.raw_datapoints.shape)
print("raw_comparisons:", model.raw_comparisons.shape)
print("supports_mll:", model.supports_mll)

mll = model.make_mll()
type(mll).__name__

## 4. Fit the preference model

`PairwiseGP` uses a Laplace approximation rather than the exact Gaussian regression likelihood used by `SingleTaskGP`.

In [ ]:
mll = model.make_mll()
fit_gpytorch_mll(mll)
model.eval()

## 5. Posterior latent utility

In [ ]:
test_X = torch.linspace(0.0, 1.0, 250, dtype=dtype).unsqueeze(-1)
with torch.no_grad():
    posterior = model.posterior(test_X)
    mean = posterior.mean.squeeze(-1)
    std = posterior.variance.sqrt().squeeze(-1)

# Pairwise utility is identifiable only up to an additive / scale convention.
# Center both curves before visual comparison.
true_u = latent_utility(test_X)
mean_centered = mean - mean.mean()
true_centered = true_u - true_u.mean()
lower = mean_centered - 1.96 * std
upper = mean_centered + 1.96 * std

In [ ]:
plt.figure(figsize=(9, 5))
plt.plot(test_X.squeeze(-1), true_centered, linestyle="--", label="true latent utility (centered)")
plt.plot(test_X.squeeze(-1), mean_centered, label="posterior mean (centered)")
plt.fill_between(test_X.squeeze(-1), lower, upper, alpha=0.2, label="95% interval")
plt.scatter(datapoints.squeeze(-1), utility - utility.mean(), s=25, alpha=0.5, label="items")
plt.xlabel("x")
plt.ylabel("relative utility")
plt.legend()
plt.title("PairwiseGP latent utility")
plt.show()

## 6. Identify the currently preferred region

The posterior mean can be used to rank candidate items even though the model was trained only on pairwise comparisons.

In [ ]:
best_idx = mean.argmax()
best_x = test_X[best_idx]
print(f"posterior-best x: {best_x.item():.4f}")

## 7. When not to use this model

If reliable absolute scalar targets are available, ordinary regression models such as `SingleTaskGP` are usually simpler and use more information per observation. `PairwiseGP` is most useful when the measurement process itself is comparative.